In [1]:
# Nếu chạy Colab L4, cell này chạy 1 lần là đủ.
!pip -q install pandas numpy scikit-learn tqdm torch transformers accelerate bitsandbytes sentence-transformers

In [2]:
!pip install -U huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [3]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer


In [4]:
# Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [5]:
TRAIN_PATH = "/content/asap_train.csv"
VAL_PATH   = "/content/asap_val.csv"
TEST_PATH  = "/content/asap_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

required_cols = ["essay_id", "essay_set", "essay", "domain1_score", "domain1_score_norm"]

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = set(required_cols) - set(df.columns)
    assert not missing, f"{name} missing columns: {missing}"

train_df.head()

,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


In [ ]:
ESSAY_SET = 2

p1_train = train_df[train_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_val   = val_df[val_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_test  = test_df[test_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)

print("P1 train:", p1_train.shape)
print("P1 val:  ", p1_val.shape)
print("P1 test: ", p1_test.shape)

p1_train[["essay_id", "essay_set", "domain1_score", "domain1_score_norm"]].head()

P1 train: (1248, 5)
P1 val:   (180, 5)
P1 test:  (355, 5)


,essay_id,essay_set,domain1_score,domain1_score_norm
0,1311,1,10.0,0.8
1,735,1,9.0,0.7
2,1152,1,8.0,0.6
3,1314,1,10.0,0.8
4,1645,1,11.0,0.9


In [7]:
ASAP_SCORE_RANGES = {
    1: (2, 12),
    2: (1, 6),
    3: (0, 3),
    4: (0, 3),
    5: (0, 4),
    6: (0, 4),
    7: (0, 30),
    8: (0, 60),
}

Y_MIN, Y_MAX = ASAP_SCORE_RANGES[ESSAY_SET]
Y_MIN, Y_MAX

(2, 12)

In [8]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    spearman = pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")

    return {
        "QWK": float(qwk),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "Spearman": float(spearman) if not pd.isna(spearman) else np.nan,
    }


In [9]:
def sample_pairs(df, num_pairs=1000, seed=42, min_per_essay=3):
    rng = np.random.default_rng(seed)
    essay_ids = df["essay_id"].astype(int).tolist()

    pairs = set()
    counts = {eid: 0 for eid in essay_ids}

    # 1. Ensure each essay appears at least min_per_essay times if possible
    attempts = 0
    max_attempts = num_pairs * 200

    while min(counts.values()) < min_per_essay and len(pairs) < num_pairs and attempts < max_attempts:
        underrepresented = [eid for eid, c in counts.items() if c < min_per_essay]

        a = int(rng.choice(underrepresented))
        b = int(rng.choice([eid for eid in essay_ids if eid != a]))

        x, y = sorted((a, b))

        if (x, y) not in pairs:
            pairs.add((x, y))
            counts[x] += 1
            counts[y] += 1

        attempts += 1

    # 2. Fill remaining pairs randomly
    while len(pairs) < num_pairs:
        a, b = rng.choice(essay_ids, size=2, replace=False)
        x, y = sorted((int(a), int(b)))
        pairs.add((x, y))

    return pd.DataFrame(list(pairs), columns=["essay_id_1", "essay_id_2"])


# Paper-like transductive setup:
# LCES main experiments score the target essay set as a whole. The scoring
# model is trained from comparisons among the essays to be scored, then the
# learned latent scores are linearly mapped to the rubric range.
p1_train_split = p1_train.copy()
p1_val_split = p1_val.copy()
p1_test_split = p1_test.copy()

p1_train_split["split"] = "train"
p1_val_split["split"] = "val"
p1_test_split["split"] = "test"

p1_all = pd.concat(
    [p1_train_split, p1_val_split, p1_test_split],
    axis=0,
    ignore_index=True
)

# Paper uses M=5,000 pairwise comparisons. You can set this to 6000 if you
# want to reuse your previous budget, but 5000 is the paper-like default.
M_PAIRWISE = 7500

all_pairs = sample_pairs(
    p1_all,
    num_pairs=M_PAIRWISE,
    seed=SEED,
    min_per_essay=4
)

print("P1 all:", p1_all.shape)
print("Pairwise comparisons M:", all_pairs.shape)
all_pairs.head()


P1 all: (1783, 6)
Pairwise comparisons M: (7500, 2)


,essay_id_1,essay_id_2
0,1453,1521
1,1255,1765
2,1109,1482
3,191,848
4,224,796


In [10]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# A100-friendly BF16 loading. If your runtime does not support BF16, switch
# torch_dtype to torch.float16. 4-bit is not needed for A100 and can be slower.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
llm.eval()

print("Loaded:", MODEL_ID)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


In [ ]:
P1_RUBRIC_COMPACT = """
Evaluate the overall quality of a persuasive essay written to a newspaper about censorship in libraries.

This essay set is assessed using two trait domains:

Domain 1: Writing Applications
Prefer the essay that better:
- fully accomplishes the persuasive writing task,
- takes and maintains a clear position on censorship in libraries,
- supports the position with relevant, convincing, and well-developed reasons, examples, observations, or reading,
- stays focused on the topic and avoids tangents,
- presents a unifying main idea or theme,
- is logically organized with a clear beginning, middle, and end,
- uses transitions and topic sentences to connect ideas,
- has an effective introduction and conclusion,
- uses precise and appropriate vocabulary,
- demonstrates fluency and sentence variety,
- adjusts language and tone appropriately for a newspaper audience,
- shows a clear sense of audience and, when possible, an original perspective.

Domain 2: Language Conventions
Also consider which essay shows better control of:
- capitalization,
- punctuation,
- spelling,
- grammar and Standard English usage,
- paragraphing,
- sentence structure,
- avoidance of run-ons and fragments.

When making the final pairwise choice, prioritize Domain 1 writing quality, but use Domain 2 language conventions as an important secondary factor.
Choose "tie" only if the two essays are genuinely very close in overall quality.
""".strip()

In [ ]:
P1_PROMPT_TEXT = """
Censorship in the Libraries

"All of us can think of a book that we hope none of our children or any other children have taken off the shelf.
But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly
the same right and so does everyone else. And then we have no books left on the shelf for any of us."
-- Katherine Paterson, Author

Write a persuasive essay to a newspaper reflecting your views on censorship in libraries.
Do you believe that certain materials, such as books, music, movies, magazines, etc.,
should be removed from the shelves if they are found offensive?

Support your position with convincing arguments from your own experience, observations, and/or reading.
""".strip()

In [13]:
SYSTEM_PROMPT_ASAP = """
As an English teacher, your primary responsibility is to evaluate the writing quality of essays written by middle school students on an English exam.
During the assessment process, you will be provided with a prompt, rubric guidelines, and two essays.
Determine which essay, Essay 1 or Essay 2, would receive a higher overall score, or respond with tie if they would receive the same score.
""".strip()


def build_pairwise_prompt(essay1, essay2):
    """Paper-like LCES prompt for ASAP pairwise comparison.

    The paper uses a teacher role, rubric, an anonymization note, brief
    reasoning, and a final JSON decision. The parser below is robust so that
    malformed JSON does not automatically collapse the pipeline.
    """
    return f"""
# Prompt
{P1_PROMPT_TEXT}

# Rubric Guidelines
{P1_RUBRIC_COMPACT}

# Note
The essays may contain anonymized entities such as PERSON, ORGANIZATION, LOCATION, DATE, TIME, MONEY, PERCENT, CAPS, or NUM.
Do not penalize the essay because of anonymization.

# Essay1
{essay1}

# Essay2
{essay2}

Provide your reasoning and final decision in JSON format:
{{"reasoning": "Your reasoning in one concise sentence.", "preference": "essay1" or "essay2" or "tie"}}
""".strip()


In [14]:
def extract_preference(text):
    """Extract preference from paper-like JSON output, with robust fallbacks."""
    raw = text
    text = text.strip()

    def normalize_pref(pref):
        pref = str(pref).lower().strip()
        pref = pref.replace(" ", "")
        pref = pref.replace("_", "")
        pref = pref.replace('"', "")
        pref = pref.replace("'", "")
        if pref in ["essay1", "1"]:
            return "essay1"
        if pref in ["essay2", "2"]:
            return "essay2"
        if pref in ["tie", "equal", "same", "samequality", "both"]:
            return "tie"
        return None

    # 1. Direct JSON parse.
    try:
        obj = json.loads(text)
        pref = normalize_pref(obj.get("preference", ""))
        if pref is not None:
            return {
                "reasoning": str(obj.get("reasoning", ""))[:500],
                "preference": pref,
                "raw_text": raw[:2000],
            }
    except Exception:
        pass

    # 2. Extract first JSON-looking block.
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            pref = normalize_pref(obj.get("preference", ""))
            if pref is not None:
                return {
                    "reasoning": str(obj.get("reasoning", ""))[:500],
                    "preference": pref,
                    "raw_text": raw[:2000],
                }
        except Exception:
            pass

    # 3. Regex preference field from imperfect JSON/text.
    pref_match = re.search(
        r'["\']?preference["\']?\s*[:=]\s*["\']?(essay\s*1|essay\s*2|essay1|essay2|tie)',
        text,
        flags=re.IGNORECASE,
    )
    if pref_match:
        pref = normalize_pref(pref_match.group(1))
        if pref is not None:
            return {"reasoning": text[:500], "preference": pref, "raw_text": raw[:2000]}

    # 4. Final-decision style fallback.
    decision_match = re.search(
        r'(decision|final decision|answer)\s*[:=\-]?\s*(essay\s*1|essay\s*2|essay1|essay2|tie)',
        text,
        flags=re.IGNORECASE,
    )
    if decision_match:
        pref = normalize_pref(decision_match.group(2))
        if pref is not None:
            return {"reasoning": text[:500], "preference": pref, "raw_text": raw[:2000]}

    # 5. Last-resort: inspect the tail only. This avoids picking up Essay1/Essay2
    # from the prompt if the model echoes part of the input.
    tail = text[-500:].lower()
    if re.search(r"\bessay\s*1\b|\bessay1\b", tail):
        pref = "essay1"
    elif re.search(r"\bessay\s*2\b|\bessay2\b", tail):
        pref = "essay2"
    elif re.search(r"\btie\b|same score|equal quality|equally", tail):
        pref = "tie"
    else:
        # Parser failure should be rare. Returning tie is still consistent with
        # LCES, but inspect raw_text if many rows fall here.
        pref = "tie"

    return {"reasoning": text[:500], "preference": pref, "raw_text": raw[:2000]}


tokenizer.padding_side = "left"

@torch.inference_mode()
def call_local_llm(prompts, max_new_tokens=64, temperature=0.1):
    """Batched local LLM inference for pairwise judgments."""
    input_texts = []

    for prompt in prompts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_ASAP},
            {"role": "user", "content": prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            input_text = SYSTEM_PROMPT_ASAP + "\n\n" + prompt

        input_texts.append(input_text)

    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(llm.device)

    do_sample = temperature is not None and temperature > 0

    output_ids = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, prompt_len:]

    texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    results = [extract_preference(t) for t in texts]

    del inputs, output_ids, generated_ids

    return results


In [15]:
# Quick smoke test
sample_essay_1 = p1_all.iloc[0]["essay"]
sample_essay_2 = p1_all.iloc[1]["essay"]

test_prompt = build_pairwise_prompt(sample_essay_1, sample_essay_2)
call_local_llm([test_prompt], max_new_tokens=64, temperature=0.1)


[{'reasoning': 'Essay 1 provides a clearer position, more detailed examples, and a better structure with transitions, making it more persuasive and coherent.',
  'preference': 'essay1',
  'raw_text': '{"reasoning": "Essay 1 provides a clearer position, more detailed examples, and a better structure with transitions, making it more persuasive and coherent.", "preference": "essay1"}'}]

In [16]:
def preference_to_label(preference):
    pref = str(preference).lower().strip()
    pref = pref.replace(" ", "")
    pref = pref.replace("_", "")
    pref = pref.replace('"', "")
    pref = pref.replace("'", "")

    if pref in ["essay1", "1"]:
        return 1.0
    if pref in ["essay2", "2"]:
        return 0.0
    if pref in ["tie", "equal", "same", "both"]:
        return 0.5

    return 0.5

def debias_label(label_forward, label_reverse):
    """
    Forward: essay_id_1 as Essay 1, essay_id_2 as Essay 2.
    Reverse: essay_id_2 as Essay 1, essay_id_1 as Essay 2.

    If consistent:
      label_forward == 1 - label_reverse

    Else:
      tie.
    """
    if np.isclose(label_forward, 1.0 - label_reverse):
        return label_forward
    return 0.5

In [17]:
def generate_pairwise_judgments(
    df,
    pairs_df,
    debias=True,
    batch_size=16,
    max_new_tokens=64,
    temperature=0.1,
):
    """Generate pairwise LLM judgments with LCES position-bias correction."""
    essay_map = df.set_index("essay_id")["essay"].to_dict()
    rows = []

    pair_rows = list(pairs_df.itertuples(index=False))

    for start in tqdm(range(0, len(pair_rows), batch_size)):
        batch = pair_rows[start:start + batch_size]

        forward_prompts = []
        reverse_prompts = []
        meta = []

        for row in batch:
            id1 = int(row.essay_id_1)
            id2 = int(row.essay_id_2)

            essay1 = essay_map[id1]
            essay2 = essay_map[id2]

            forward_prompts.append(build_pairwise_prompt(essay1, essay2))

            if debias:
                reverse_prompts.append(build_pairwise_prompt(essay2, essay1))

            meta.append((id1, id2))

        forward_results = call_local_llm(
            forward_prompts,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )

        if debias:
            reverse_results = call_local_llm(
                reverse_prompts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )
        else:
            reverse_results = [None] * len(forward_results)

        for (id1, id2), result_forward, result_reverse in zip(meta, forward_results, reverse_results):
            label_forward = preference_to_label(result_forward.get("preference", "tie"))

            if debias:
                label_reverse = preference_to_label(result_reverse.get("preference", "tie"))
                final_label = debias_label(label_forward, label_reverse)

                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": final_label,
                    "pref_forward": result_forward.get("preference", "tie"),
                    "pref_reverse": result_reverse.get("preference", "tie"),
                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": result_reverse.get("reasoning", ""),
                    "raw_forward": result_forward.get("raw_text", ""),
                    "raw_reverse": result_reverse.get("raw_text", ""),
                })
            else:
                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": label_forward,
                    "pref_forward": result_forward.get("preference", "tie"),
                    "pref_reverse": None,
                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": None,
                    "raw_forward": result_forward.get("raw_text", ""),
                    "raw_reverse": None,
                })

        del batch, forward_prompts, reverse_prompts, forward_results, reverse_results, meta

        batch_idx = start // batch_size
        if torch.cuda.is_available() and batch_idx % 25 == 0:
            torch.cuda.empty_cache()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def inspect_judgments(judgments):
    print("pref_forward")
    print(judgments["pref_forward"].value_counts(dropna=False))

    if "pref_reverse" in judgments.columns:
        print("\npref_reverse")
        print(judgments["pref_reverse"].value_counts(dropna=False))

    print("\nlabel")
    print(judgments["label"].value_counts(dropna=False))

    tie_rate = (judgments["label"] == 0.5).mean()
    print(f"\nTie rate: {tie_rate:.2%}")

    if tie_rate > 0.60:
        raise ValueError("Tie rate is high. Inspect raw outputs before training RankNet.")


### Pairwise judgment generation
The following cell generates or loads cached LCES pairwise comparisons for the transductive Prompt 1 essay set.


In [ ]:
JUDGMENT_CACHE = "all_judgments_p2_qwen25_7b_paperprompt_m5000_debias.csv"

if os.path.exists(JUDGMENT_CACHE):
    all_judgments = pd.read_csv(JUDGMENT_CACHE)
    print("Loaded cached judgments:", JUDGMENT_CACHE)
else:
    all_judgments = generate_pairwise_judgments(
        df=p1_all,
        pairs_df=all_pairs,
        debias=True,
        batch_size=16,
        max_new_tokens=64,
        temperature=0.1,
    )

    all_judgments.to_csv(JUDGMENT_CACHE, index=False)
    print("Saved judgments:", JUDGMENT_CACHE)

inspect_judgments(all_judgments)

# Free LLM VRAM before loading the embedding model and training RankNet.
if "llm" in globals():
    del llm
if "tokenizer" in globals():
    del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


  0%|          | 0/469 [00:00<?, ?it/s]

Saved judgments: all_judgments_p1_qwen25_7b_paperprompt_m5000_debias.csv
pref_forward
pref_forward
essay2    5620
essay1    1875
tie          5
Name: count, dtype: int64

pref_reverse
pref_reverse
essay2    5684
essay1    1810
tie          6
Name: count, dtype: int64

label
label
0.5    3857
1.0    1853
0.0    1790
Name: count, dtype: int64

Tie rate: 51.43%


In [20]:
used_ids = set(all_judgments["essay_id_1"]) | set(all_judgments["essay_id_2"])

print("Total P1 essays:", len(p1_all))
print("Essays appearing in comparisons:", len(used_ids))
print("Coverage:", len(used_ids) / len(p1_all))

pair_count = pd.concat([
    all_judgments["essay_id_1"],
    all_judgments["essay_id_2"],
]).value_counts()

print("\nPair count per essay:")
print(pair_count.describe())


Total P1 essays: 1783
Essays appearing in comparisons: 1783
Coverage: 1.0

Pair count per essay:
count    1783.000000
mean        8.412787
std         2.175028
min         4.000000
25%         7.000000
50%         8.000000
75%        10.000000
max        17.000000
Name: count, dtype: float64


In [21]:
# Paper main experiments use OpenAI text-embedding-3-large. To keep this
# notebook runnable without an API key, we use RoBERTa-base [CLS], which the
# paper also reports as a robust alternative in Appendix B.
EMBEDDING_MODEL = "roberta-base"

embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL).to(DEVICE)
embedding_model.eval()

@torch.inference_mode()
def encode_essays_cls(df, text_col="essay", batch_size=16, max_length=512):
    all_vecs = []
    texts = df[text_col].tolist()

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        inputs = embedding_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(DEVICE)

        outputs = embedding_model(**inputs)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        all_vecs.append(cls_vec.detach().cpu().numpy())

        del inputs, outputs, cls_vec

    return np.vstack(all_vecs).astype(np.float32)


all_embeddings = encode_essays_cls(p1_all, batch_size=16, max_length=512)

print("Embedding model:", EMBEDDING_MODEL)
print("all_embeddings:", all_embeddings.shape)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/112 [00:00<?, ?it/s]

Embedding model: roberta-base
all_embeddings: (1783, 768)


In [ ]:
class RankNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()

        self.scorer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.scorer(x).squeeze(-1)

In [23]:
def train_ranknet(
    df,
    embeddings,
    judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
    device=DEVICE
):
    essay_ids = df["essay_id"].astype(int).tolist()
    id_to_idx = {eid: idx for idx, eid in enumerate(essay_ids)}

    pair_df = judgments[
        judgments["essay_id_1"].isin(id_to_idx) &
        judgments["essay_id_2"].isin(id_to_idx)
    ].copy()

    assert len(pair_df) > 0, "No valid pairwise judgments for this dataframe."

    idx_i = pair_df["essay_id_1"].map(id_to_idx).values
    idx_j = pair_df["essay_id_2"].map(id_to_idx).values
    labels = pair_df["label"].values.astype(np.float32)

    x = torch.tensor(embeddings, dtype=torch.float32, device=device)

    model = RankNet(
        input_dim=embeddings.shape[1],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    loss_fn = nn.BCELoss()

    n = len(pair_df)

    for epoch in range(1, epochs + 1):
        order = np.random.permutation(n)
        epoch_losses = []

        model.train()

        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]

            bi = torch.tensor(idx_i[batch], dtype=torch.long, device=device)
            bj = torch.tensor(idx_j[batch], dtype=torch.long, device=device)
            y = torch.tensor(labels[batch], dtype=torch.float32, device=device)

            s_i = model(x[bi])
            s_j = model(x[bj])

            pred = torch.sigmoid(s_i - s_j)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | loss = {np.mean(epoch_losses):.4f}")

    return model

In [24]:
# Paper hyperparameters: epochs=100, batch_size=4096, hidden_dim=256,
# dropout=0.3, weight_decay=0.01, lr=0.001.
ranknet = train_ranknet(
    df=p1_all,
    embeddings=all_embeddings,
    judgments=all_judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
)


Epoch 001 | loss = 0.6928
Epoch 010 | loss = 0.6660
Epoch 020 | loss = 0.6282
Epoch 030 | loss = 0.6053
Epoch 040 | loss = 0.5931
Epoch 050 | loss = 0.5855
Epoch 060 | loss = 0.5799
Epoch 070 | loss = 0.5773
Epoch 080 | loss = 0.5759
Epoch 090 | loss = 0.5750
Epoch 100 | loss = 0.5742


In [25]:
@torch.no_grad()
def predict_latent_scores(model, embeddings, device=DEVICE):
    model.eval()
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)
    scores = model(x).detach().cpu().numpy()
    return scores


all_latent = predict_latent_scores(ranknet, all_embeddings)

print("latent min:", all_latent.min())
print("latent max:", all_latent.max())
print("latent std:", all_latent.std())
print("latent head:", all_latent[:10])


latent min: -2.3388028
latent max: 1.0566946
latent std: 0.5971147
latent head: [ 0.45451796  0.30387527 -0.3573646   0.02566714  0.69406945 -1.3491533
  0.4532671  -0.04485255 -0.97178996 -0.3904624 ]


In [26]:
def fit_latent_mapper(latent_scores_ref, y_min, y_max):
    """Paper's linear latent-score conversion to the rubric range."""
    s_min = float(np.min(latent_scores_ref))
    s_max = float(np.max(latent_scores_ref))

    def mapper(latent_scores):
        latent_scores = np.asarray(latent_scores, dtype=float)

        if np.isclose(s_min, s_max):
            mapped = np.full_like(latent_scores, fill_value=(y_min + y_max) / 2)
        else:
            mapped = (latent_scores - s_min) / (s_max - s_min)
            mapped = mapped * (y_max - y_min) + y_min

        mapped = np.rint(mapped)
        mapped = np.clip(mapped, y_min, y_max)

        return mapped.astype(int)

    return mapper


score_mapper = fit_latent_mapper(all_latent, Y_MIN, Y_MAX)
all_pred = score_mapper(all_latent)

p1_all_scored = p1_all[[
    "essay_id",
    "split",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm",
]].copy()

p1_all_scored["latent_score"] = all_latent
p1_all_scored["pred_score"] = all_pred
p1_all_scored["abs_error"] = np.abs(
    p1_all_scored["domain1_score"] - p1_all_scored["pred_score"]
)

print(pd.Series(all_pred).value_counts().sort_index())
p1_all_scored.head()


2       4
3      15
4      60
5     127
6     229
7     368
8     363
9     331
10    214
11     64
12      8
Name: count, dtype: int64


,essay_id,split,essay_set,essay,domain1_score,domain1_score_norm,latent_score,pred_score,abs_error
0,1311,train,1,Dear @CAPS1 @CAPS2 @CAPS3: @CAPS4 people are s...,10.0,0.8,0.454518,10,0.0
1,735,train,1,"As you can see, technology these days is alway...",9.0,0.7,0.303875,10,1.0
2,1152,train,1,"Dear local newspaper, I am writing to you toda...",8.0,0.6,-0.357365,8,0.0
3,1314,train,1,"Dear @ORGANIZATION1, Have you ever trully marv...",10.0,0.8,0.025667,9,1.0
4,1645,train,1,"Dear @ORGANIZATION2, @DATE1, computers are a b...",11.0,0.9,0.694069,11,0.0


In [27]:
metrics_rows = []

for split in ["train", "val", "test", "all"]:
    if split == "all":
        df_eval = p1_all_scored
    else:
        df_eval = p1_all_scored[p1_all_scored["split"] == split]

    metrics_rows.append({
        "split": split,
        **compute_metrics(
            y_true=df_eval["domain1_score"],
            y_pred=df_eval["pred_score"],
        ),
    })

pd.DataFrame(metrics_rows)


,split,QWK,MAE,RMSE,Spearman
0,train,0.587131,1.221154,1.594763,0.678415
1,val,0.604263,1.272222,1.654959,0.709055
2,test,0.533687,1.295775,1.690070,0.633139
3,all,0.578588,1.241167,1.620293,0.672876


In [28]:
p1_test_results = p1_all_scored[p1_all_scored["split"] == "test"].copy()

p1_test_results[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
]].head(20)


,essay_id,domain1_score,pred_score,abs_error,latent_score
1428,1723,9.0,7,2.0,-0.722337
1429,1334,8.0,7,1.0,-0.756953
1430,966,7.0,7,0.0,-0.487273
1431,260,7.0,7,0.0,-0.574067
1432,567,8.0,5,3.0,-1.406435
1433,791,8.0,5,3.0,-1.246645
1434,230,9.0,9,0.0,0.105467
1435,1319,9.0,8,1.0,-0.414018
1436,816,9.0,7,2.0,-0.585813
1437,121,8.0,4,4.0,-1.586042


In [29]:
# Worst predictions on the held-out test split
p1_test_results.sort_values("abs_error", ascending=False)[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
    "essay",
]].head(10)


,essay_id,domain1_score,pred_score,abs_error,latent_score,essay
1720,512,8.0,3,5.0,-1.845544,Dear People Computers are not a really good th...
1630,1101,8.0,4,4.0,-1.569830,"Dear, @CAPS1 I am writting this article on com..."
1484,439,10.0,6,4.0,-0.833793,What I am going to tell you could change the w...
1477,1207,8.0,12,4.0,0.900818,"Dear editor, Have you ever used a computer? Od..."
1486,491,8.0,4,4.0,-1.564406,"Dear, @CAPS1 Who would'nt agree that more and ..."
1468,1721,9.0,5,4.0,-1.436722,"Dear @CAPS1, I believe that computers are good..."
1714,545,2.0,6,4.0,-0.837088,I think that computers are amazing. Computers ...
1660,151,8.0,4,4.0,-1.591784,Dear local newspaper I dont agree with the pe...
1682,1691,11.0,7,4.0,-0.531613,"Dear @LOCATION2, @CAPS1 your workers wake up i..."
1605,901,8.0,4,4.0,-1.577487,"Dear local news paper, In @CAPS1 oppinion I th..."
